# LABORATORIO N° 4: Diseño de algoritmos paralelos. Descomposición, particionamiento y pseudocódigo paralelo.

INTEGRANTES: Benjamín Aballay / Felipe Martínez

Asignatura: Computación paralela y distribuida(INFB8090)

Docente: Michael Miranda

Sección: 412

Fecha: 30 de Abril

Institución: Universidad Tecnológica Metropolitana

## DESAFÍO 1: Analizar estructuralmente un problema antes de paralelizar.

### 1.A) **Identificación de descomposición y dependencias**

1. **Una transformación elemento a elemento sobre un arreglo grande**

      Para este caso se escogío el problema: Ajuste de brillo en un dataset(Procesamiento de Imágenes - Data Augmentation).

*    Definición clara de entrada, salida y operación principal   

      Entrada: Un arreglo unidimensional $X$ de tamaño $N$, donde cada elemento representa el valor de un canal de color de un píxel (un número entero entre 0 y 255) perteneciente a un lote masivo de imágenes.

      Salida: Un arreglo unidimensional $Y$ de tamaño $N$, que contiene los valores de los píxeles ya modificados.

      Operación principal: A cada elemento se le suma una constante de brillo $\Delta$. Si el resultado supera el límite máximo del color, se ajusta a 255 (operación clamp). La fórmula matemática es: $Y[i] = \min(X[i] + \Delta, 255)$.
*   Tipo de descomposición   

      Conviene estrictamente una descomposición por datos. Como la misma operación se aplica a todos los elementos por igual, la estructura del problema invita a tomar el arreglo gigante y particionarlo en sub-arreglos o chunks, asignando cada bloque a un worker distinto.  

*  Unidades que podrían ejecutarse en paralelo  

      Las unidades de trabajo paralelizables son cada evaluación individual del arreglo. La transformación del píxel en el índice $i$ es una unidad computacional que puede ejecutarse de manera completamente aislada y simultánea a la transformación del píxel en el índice $i+1$.


*   Identificación de dependencias

      Este es un problema embarrassingly parallel.
       No existen dependencias de datos.  
       No hay dependencias verdaderas (Read-After-Write), ya que ningún cálculo necesita el resultado de otro.
       No hay anti-dependencias (Write-After-Read), porque leemos de $X$ y escribimos en una posición independiente de $Y$.No hay output-dependencias (Write-After-Write), porque cada posición de memoria de salida $Y[i]$ es escrita por un único hilo de ejecución.

*   Patrón de diseño

      El patrón corresponde directamente a un esquema map. Estamos "mapeando" o proyectando una función escalar independiente sobre cada elemento de un conjunto de datos, sin que los datos interactúen entre sí.



*   Parte secuencial o requerimientos de coordinación

      El cálculo computacional en sí no requiere coordinación ni barreras lógicas durante su ejecución. Sin embargo, la parte secuencial o de coordinación se remite a:  

      La división y asignación inicial del trabajo (enviar los chunks de memoria a cada worker o hilo).  
          
      El ensamblaje final , donde el hilo principal debe esperar a que todos los workers terminen para consolidar los chunks procesados en el arreglo de salida $Y$ definitivo.  

2. **Un cálculo agregativo sobre un arreglo, como suma, máximo, promedio o varianza**

      Para este caso se escogío el problema: Búsqueda de la predicción más segura (Máximo).

*    Definición clara de entrada, salida y operación principal   

      Entrada: Un arreglo unidimensional $X$ de tamaño $N$, que contiene valores de coma flotante entre $0.0$ y $1.0$. Estos valores representan el nivel de confianza (confidence score) de todas las cajas delimitadoras detectadas en un lote masivo de imágenes.

      Salida: Un único valor escalar $Y$, que corresponde al puntaje de confianza más alto de todo el arreglo.

      Operación principal: La operación es una búsqueda del valor máximo sobre el conjunto de datos: $Y = \max(X[0], X[1], ..., X[N-1])$.

*   Tipo de descomposición   

      Conviene utilizar una descomposición por datos. El arreglo gigante de puntajes se divide en sub-arreglos (o chunks). A cada proceso o hilo se le asigna uno de estos segmentos para que encuentre el valor máximo de forma aislada dentro de su porción asignada.  

*  Unidades que podrían ejecutarse en paralelo  

      Las unidades de trabajo paralelizables son las búsquedas locales de máximos dentro de cada sub-arreglo. Es decir, mientras el worker 1 busca el máximo en la primera mitad de los datos, el worker 2 puede estar buscando el máximo en la segunda mitad de forma simultánea.


*   Identificación de dependencias

      Durante la fase de cómputo local (cuando cada hilo revisa su propio chunk), no hay dependencias de lectura ni de escritura.

      Sin embargo, en la fase de consolidación, existe un riesgo de output-dependencia (condición de carrera)  si todos los workers intentan escribir o actualizar la misma variable global maximo_global al mismo tiempo. Para evitar esto, cada worker debe almacenar su resultado en una variable local temporal.

*   Patrón de diseño

      Este problema corresponde exactamente a un patrón de reduction (reducción). Tomamos un conjunto grande de datos y, mediante una operación asociativa (como es la búsqueda del máximo), lo "reducimos" a un solo resultado final.



*   Parte secuencial o requerimientos de coordinación

      A diferencia de las transformaciones 1 a 1, este algoritmo requiere coordinación estricta al final.

      Una vez que cada worker encuentra su máximo local, debe existir un punto de sincronización (como una barrera lógica)  donde todos se detengan y compartan sus resultados. La parte secuencial (o de coordinación global) consiste en tomar los máximos locales reportados por cada worker y compararlos entre sí para ensamblar el máximo global definitivo.


### 1.B) **Comparación entre esquemas de particionamiento**

*   *Uniforme*: Se divide el arreglo en $P$ partes de tamaño exactamente igual (asumiendo que $N$ es divisible por $P$).

*   *Por bloques*: Se asignan trozos contiguos a cada worker (gestionando los restos si la división no es exacta).

*   *Cíclico*: Se reparten los elementos como si se tratara de una baraja de cartas (el worker 0 procesa el índice 0, el worker 1 procesa el índice 1, y se repite el ciclo).

* *Problema utilizado*: Búsqueda de la predicción más segura

**1. ¿Qué esquema ofrece mejor balance de carga?**

  En este problema en particular, la operación matemática (evaluar if x[i] > max_local) toma exactamente el mismo tiempo de CPU para cualquier dato. Dado que la carga es completamente regular, los esquemas Uniforme y Por bloques ofrecen un balance de carga perfecto. Cada worker recibe la misma cantidad de elementos a comparar y, por lo tanto, todos deberían terminar de escanear su sector al mismo tiempo. El cíclico también balancearía bien la cantidad, pero no aporta una ventaja real aquí.  

**2. ¿Cuál favorece más la locality o acceso contiguo a datos?**

  El esquema Por bloques (y el Uniforme) es el ganador indiscutido en este aspecto. Los arreglos unidimensionales se almacenan en bloques físicos contiguos dentro de la memoria RAM. Al asignar bloques, cada worker carga un segmento completo a su memoria caché (aprovechando la localidad espacial) y lo recorre secuencialmente a máxima velocidad. Por el contrario, el esquema Cíclico destruye la locality, ya que obliga a cada worker a dar saltos en la memoria, lo que genera constantes cache misses y ralentiza la ejecución.  

**3. ¿Cuál sería más sensible a overhead de coordinación?**

  El esquema Cíclico introduciría el mayor overhead o sobrecarga. En la búsqueda de un máximo, la coordinación entre workers solo ocurre al final (la reducción global). Sin embargo, el esquema cíclico obliga a realizar cálculos extra (operaciones de módulo como i % P) o a gestionar la lógica de saltos en cada iteración del ciclo for para saber qué dato le toca a qué worker. Los esquemas por bloques solo calculan los índices de inicio y fin una única vez al comienzo.  

**4. ¿Qué ocurre si el costo por dato o tarea no es uniforme?**

  Si el problema mutara y el costo dejara de ser regular (por ejemplo, si al encontrar un puntaje alto el algoritmo debiera ejecutar tareas de procesamiento pesado, como decodificar una máscara de segmentación tipo Segment Anything Model, y todos los objetos detectados estuvieran agrupados al principio del arreglo) , el esquema Por bloques fallaría rotundamente. Un solo worker recibiría toda la carga pesada mientras los demás quedarían ociosos. En ese caso hipotético, el particionamiento Cíclico sería mucho mejor, ya que distribuiría probabilísticamente el trabajo pesado entre todos los hilos, equilibrando la carga final.





## DESAFÍO 2: Pasar del problema al pseudocódigo paralelo.

### 2.A) **Caso de arreglo independiente con reducción**

1) *Definicion del problema*

*   **Contexto:**

      Un modelo de visión computacional ha procesado un lote de imágenes y ha generado un arreglo unidimensional masivo.

*   **Entrada:**

      Un arreglo X de tamaño N, donde cada elemento es un número de coma flotante (entre 0.0 y 1.0) que representa el nivel de confianza (confidence score) de una detección.

*   **Salida:**

      Un único valor flotante que corresponde al puntaje máximo encontrado en todo el arreglo.


*   **Operacion:**

      Escanear el arreglo dividiéndolo en chunks (particiones), encontrar el máximo local en cada uno de ellos simultáneamente y, finalmente, comparar esos máximos locales para obtener el máximo global absoluto.


2) *Pseudocodigo secuencial*

FUNCION buscar_maximo_secuencial(X, N):
    maximo = 0.0
    
    PARA i DESDE 0 HASTA N - 1:
        SI X[i] > maximo:
            maximo = X[i]
            
    RETORNAR maximo

3) *Pseudocodigo paralelo*


// Variables compartidas por todos los hilos

VARIABLES GLOBALES:

    X: Arreglo de N flotantes
    maximo_global: Flotante = 0.0
    candado_maximo: Mutex (Cerrojo lógico)

// Función que ejecuta cada hilo independiente

FUNCION worker_buscar_maximo(inicio, fin):

    // Cada worker tiene su propia memoria temporal
    maximo_local = 0.0
    
    // 1. Fase de Trabajo Local (Sin dependencias)
    PARA i DESDE inicio HASTA fin - 1:
        SI X[i] > maximo_local:
            maximo_local = X[i]
            
    // 2. Fase de Reducción / Combinación (Punto de Sincronización)
    BLOQUEAR(candado_maximo)
    SI maximo_local > maximo_global:
        maximo_global = maximo_local
    DESBLOQUEAR(candado_maximo)


// Función controladora (Hilo principal)


FUNCION principal_paralela(X, N, cantidad_workers):

    tamaño_chunk = N / cantidad_workers
    hilos = Nuevo Arreglo de tamaño cantidad_workers
    
    // Asignación de trabajo
    PARA w DESDE 0 HASTA cantidad_workers - 1:
        inicio = w * tamaño_chunk
        fin = inicio + tamaño_chunk
        // Lanzar hilo de forma asíncrona
        hilos[w] = INICIAR_HILO(worker_buscar_maximo, inicio, fin)
        
    // Barrera: Esperar a que todos los workers terminen
    ESPERAR_A_TODOS(hilos)
    
    RETORNAR maximo_global

4) *Especificación de variables locales y globales*

**Variables Globales (Compartidas):**

* X: El arreglo con los datos. Es de solo lectura durante la ejecución paralela, por lo que no genera conflictos.

* maximo_global: Almacena el resultado final. Como múltiples workers podrían intentar sobrescribirla a la vez (output-dependencia), es una zona de riesgo.

* candado_maximo (Mutex): Mecanismo de sincronización que protege a maximo_global de escrituras simultáneas.

**Variables Locales (Privadas de cada worker):**

* inicio y fin: Definen el límite de iteración (el chunk) exclusivo de cada worker.

* i: El iterador del ciclo PARA.

* maximo_local: Almacena el mejor resultado encontrado por el worker dentro de su chunk asignado, evitando tener que coordinarse con los demás durante la fase de cálculo pesada.

5) *Explicación de la reducción o combinación final*

La reducción final ocurre en las últimas líneas de la función del worker. En lugar de que cada worker modifique la variable global maximo_global en cada iteración de su ciclo (lo cual crearía un cuello de botella terrible), primero computan su maximo_local de manera aislada.  

Una vez que terminan su bloque, deben actualizar el maximo_global. Para evitar una condición de carrera (donde dos workers intentan escribir al mismo tiempo y se corrompe el dato), se implementa una sección crítica usando un candado lógico (Mutex). Un worker adquiere el candado, compara su máximo local con el global, lo actualiza si es mayor, y luego libera el candado para que el siguiente worker pueda hacer lo mismo. Finalmente, el hilo principal espera a todos los workers en una barrera lógica antes de retornar el valor.

6) *Justificación del esquema de particionamiento elegido*

Se utiliza un particionamiento por bloques  (asignando trozos contiguos del arreglo a cada worker mediante los índices inicio y fin). Esta es la estrategia óptima para este problema por dos motivos:  


* Balance de carga regular: La operación condicional (SI X[i] > maximo) tiene un costo computacional idéntico para cualquier dato. Por lo tanto, dividir el arreglo en bloques de igual tamaño garantiza que todos los workers tengan exactamente la misma cantidad de trabajo.  

* Locality (Localidad Espacial): Los arreglos se almacenan en espacios de memoria contiguos. Al usar bloques, cada worker lee datos secuenciales. Esto permite que la memoria caché del procesador pre-cargue los datos eficientemente (maximizando los aciertos de caché), lo cual sería imposible con un esquema cíclico que obligaría a dar saltos aleatorios en la memoria.

### 2.B) **Caso con dependencia local o frontera**

1) *Identificación precisa de la dependencia*

En este problema existe una dependencia espacial local. Para calcular el nuevo valor en la posición de salida $Y[i]$, el algoritmo necesita leer estrictamente tres posiciones del arreglo de entrada: $X[i-1]$ (vecino izquierdo), $X[i]$ (valor central) y $X[i+1]$ (vecino derecho).

2) *Por qué una partición ingenua es incorrecta*

Si hacemos un particionamiento "ingenuo" y cortamos el arreglo en bloques cerrados y aislados, se produce un error en los bordes de cada bloque. Por ejemplo, si al Worker 2 le asignamos calcular desde el índice 50 hasta el 99, cuando intente calcular su primer valor ($Y[50]$), necesitará leer $X[49]$. Pero si su partición es estrictamente [50, 99], no tendrá acceso al dato 49 (que le pertenece al Worker 1). Esto provocaría un fallo de acceso a memoria (o un cálculo incompleto) justo en las fronteras de los chunks.

3) *Manejo de fronteras o halos conceptuales*

Para solucionar esto, se utiliza el concepto de celdas fantasma (ghost cells) o halos.Conceptualmente, la partición de escritura es diferente a la partición de lectura:Partición de escritura (mutuamente excluyente): El Worker es dueño exclusivo de escribir en $Y$ desde inicio hasta fin "- 1".Partición de lectura (con halos solapados): El Worker tiene permiso para leer en $X$ desde inicio "- 1" hasta fin (es decir, un elemento más a la izquierda y uno más a la derecha). Como la lectura simultánea no genera condiciones de carrera, los workers comparten lógicamente esos datos de los bordes.


4) *Pseudocodigo Paralelo*

// Variables globales compartidas

VARIABLES GLOBALES:

    X: Arreglo de entrada de tamaño N (Solo lectura)
    Y: Arreglo de salida de tamaño N (Escritura)

// Función que ejecuta cada worker

FUNCION worker_stencil_1d(inicio, fin, N):
    
    PARA i DESDE inicio HASTA fin - 1:
        // Caso Frontera Izquierda Absoluta (borde del arreglo total)
        SI i == 0:
            Y[i] = (X[0] + X[1]) / 2.0
            
        // Caso Frontera Derecha Absoluta (borde del arreglo total)
        SINO SI i == N - 1:
            Y[i] = (X[N-2] + X[N-1]) / 2.0
            
        // Caso General (incluye los halos internos entre particiones)
        SINO:
            Y[i] = (X[i-1] + X[i] + X[i+1]) / 3.0


// Función controladora

FUNCION principal_stencil_paralelo(N, cantidad_workers):

    tamaño_chunk = N / cantidad_workers
    hilos = Nuevo Arreglo
    
    // Reparto de trabajo
    PARA w DESDE 0 HASTA cantidad_workers - 1:
        inicio = w * tamaño_chunk
        fin = inicio + tamaño_chunk
        hilos[w] = INICIAR_HILO(worker_stencil_1d, inicio, fin, N)
        
    // Punto de sincronización global
    ESPERAR_A_TODOS(hilos)
    
    RETORNAR Y

5) *Requisito de barreras lógicas, etapas o combinación*

Dado que el problema se resuelve escribiendo los resultados en un arreglo de salida distinto ($Y$) al de lectura ($X$), no hay dependencias de datos entre los workers durante la ejecución. Por lo tanto, no se necesitan barreras lógicas mientras ocurre el cálculo.

Sin embargo, sí se requiere una barrera lógica estricta al final. El hilo principal debe ejecutar un ensamblaje final (representado por ESPERAR_A_TODOS en el código) para garantizar que todos los workers hayan terminado de escribir sus porciones con halos antes de que el programa pueda utilizar el arreglo $Y$ suavizado de forma segura.

## DESAFÍO 3: Diseño integrador de una estrategia paralela.

### 3.A) **Diseño de una estrategia para tareas independientes o pipeline simple**

1)  *Descripción clara del problema*

Se cuenta con un dataset de $100.000$ imágenes de especies marinas chilenas en alta resolución. Para entrenar un modelo de Deep Learning, cada imagen debe pasar por un proceso de preprocesamiento estándar: redimensionamiento a $224 \times 224$ píxeles, conversión a escala de grises y normalización de valores. Como la transformación de una imagen es totalmente ajena a las demás, el problema se aborda como un conjunto de tareas independientes.

2) *Identificación de entrada, salida y subtareas Entrada*

Una lista de rutas de archivos (strings) apuntando a las imágenes originales en disco.Salida: Archivos procesados almacenados en un directorio de destino.Subtareas:Lectura de archivo desde disco (I/O).Decodificación de la imagen a matriz numérica.Aplicación de filtros y redimensionamiento (Cómputo).Guardado del resultado en disco (I/O).

3) *Estrategia de asignación del trabajo*

Se utilizará un esquema de Worker Pool (o lote de hilos). Dado que el número de imágenes ($N$) es mucho mayor que el número de procesadores ($P$), no se crea un hilo por imagen. En su lugar, se divide el lote total en "paquetes" o chunks de tamaño fijo (por ejemplo, 500 imágenes por paquete) y se asignan dinámicamente a los workers disponibles.

4) *Análisis de balance de carga*

Aunque la operación matemática es similar, el tiempo de ejecución puede variar según el tamaño original de cada archivo (una imagen de 4K tarda más en decodificarse que una de 720p). Un particionamiento estático por bloques podría dejar a algunos workers ociosos si les tocan todas las imágenes pesadas. Por ello, se propone una asignación dinámica donde cada worker solicita un nuevo paquete de la cola global solo cuando termina el anterior, asegurando que todos los núcleos trabajen hasta el final.

5) *Identificación de puntos de coordinación*

Se requiere un punto de coordinación seguro (Thread-safe Queue) para que los workers extraigan las rutas de las imágenes sin repetir trabajo.Contador de progreso: Una variable global protegida por un candado (Mutex) para llevar el registro de cuántas imágenes han sido procesadas exitosamente.Barrera final: El programa principal debe esperar a que todos los hilos del pool terminen antes de reportar el éxito de la operación.

6) *Discusión del costo de comunicación o transferencia de datos*

En este diseño, el principal overhead no es el cómputo, sino el cuello de botella de Entrada/Salida (I/O). Si demasiados hilos intentan leer del mismo disco mecánico simultáneamente, el rendimiento cae. La transferencia de datos consiste en pasar la ruta del archivo (un string pequeño) al worker, pero el worker asume el costo pesado de cargar la matriz de la imagen a su memoria local.

7) *Pseudocodigo Paralelo*

// Variables Globales

COLA_TAREAS = Lista de todas las rutas de imágenes

CONTADOR_EXITO = 0

CERROJO_CONTADOR = Mutex()

FUNCION procesador_de_imagenes():

    MIENTRAS COLA_TAREAS no esté vacía:
        // Coordinación para extraer tarea
        ruta = COLA_TAREAS.extraer_siguiente()
        
        SI ruta es válida:
            // Tarea independiente (Cómputo pesado)
            img = LEER_IMAGEN(ruta)
            img_procesada = REDIMENSIONAR_Y_FILTRAR(img)
            GUARDAR(img_procesada)
            
            // Coordinación para actualizar estado
            BLOQUEAR(CERROJO_CONTADOR)
            CONTADOR_EXITO = CONTADOR_EXITO + 1
            DESBLOQUEAR(CERROJO_CONTADOR)

FUNCION principal():

    P = OBTENER_NUMERO_NUCLEOS()
    POOL_HILOS = Arreglo de tamaño P
    
    // Lanzar Workers
    PARA i DESDE 0 HASTA P - 1:
        POOL_HILOS[i] = INICIAR_HILO(procesador_de_imagenes)
        
    // Punto de sincronización global
    ESPERAR_A_TODOS(POOL_HILOS)
    IMPRIMIR("Proceso completado. Imágenes procesadas:", CONTADOR_EXITO)

### 3.B) **Justificación técnica final del diseño**

Frente a un nuevo problema computacional basado en arreglos o conjuntos de tareas, el criterio fundamental de diseño no radica en la selección inmediata de una herramienta o librería concurrente, sino en el análisis estructural previo para evitar la simple paralelización ciega del código secuencial.

El primer paso es evaluar la naturaleza de los datos y la presencia de dependencias. Si nos enfrentamos a un problema puramente embarrassingly parallel, como el ajuste de brillo masivo analizado en el primer desafío, la descomposición natural es por datos, ya que no existen colisiones de escritura. Por el contrario, si la operación exige leer posiciones adyacentes —como en el filtro espacial unidimensional (stencil)—, el particionamiento debe contemplar el manejo de celdas fantasma o halos en las fronteras. Asimismo, es vital identificar tempranamente las dependencias de salida (output-dependencies); en el cálculo agregativo del puntaje máximo de confianza, la dependencia global obligó a confinar el cálculo en memorias locales antes de intentar cualquier escritura compartida.

El segundo criterio rector es la regularidad de la carga y la granularidad. Cuando el costo computacional es idéntico para cada elemento (carga regular), un particionamiento estático por bloques es óptimo, ya que maximiza la localidad espacial de la memoria caché. Sin embargo, si la carga es netamente irregular —como en el procesamiento de un dataset de especies marinas extraído de FathomNet, donde las resoluciones de las imágenes y los tiempos de lectura varían drásticamente— forzar bloques contiguos expone al sistema a los graves riesgos de una estrategia mal balanceada. El principal peligro es la inanición parcial (starvation): múltiples hilos ociosos esperando a que un único trabajador termine su pesada carga, destruyendo el speedup general. En escenarios I/O-bound o de carga heterogénea, un modelo dinámico de worker pool ajusta mucho mejor la granularidad operativa.

En tercer lugar, se debe presupuestar el costo esperado de coordinación y la necesidad de reducción o barreras. La regla de diseño es que toda coordinación penaliza el rendimiento por el overhead introducido. Todo punto de sincronización (como el Mutex utilizado para ensamblar el máximo global o el cerrojo para actualizar el contador de tareas exitosas) debe aislarse estrictamente fuera de los ciclos intensivos de cómputo (hot loops). Las barreras lógicas solo deben existir al finalizar etapas o en el ensamblaje definitivo.

Finalmente, al evaluar las limitaciones del diseño propuesto, es imperativo reconocer que estrategias como el particionamiento por bloques asumen que la memoria RAM del sistema puede albergar simultáneamente los chunks asignados. Una limitación estructural de nuestros enfoques es que, ante colecciones masivas que excedan la memoria principal, se requeriría orquestar un procesamiento por flujos continuos (streaming), lo cual sumaría una latencia considerable al overhead de transferencia desde el disco.

En conclusión, un diseño paralelo técnicamente robusto emerge de equilibrar dinámicamente la granularidad del trabajo con el mínimo costo de comunicación posible, respetando sin excepciones las restricciones matemáticas impuestas por la estructura del problema.